In [0]:
df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/train/train_schema/train_data/train.csv")
df.show(5)

In [0]:
df.count()

In [0]:
# from pyspark.sql.functions import sum
# df.select(
#     sum("sales").alias("Total_sales"),
#     sum("profit").alias("total_profit")
# ).show()

In [0]:
from pyspark.sql.functions import col

df.filter(~col("sales").rlike("^[0-9.]+$")) \
  .select("sales") \
  .distinct() \
  .show()


In [0]:
from pyspark.sql.functions import expr, col

df1 = df.withColumn(
    "sales_num",
    expr("try_cast(sales as double)")
)


In [0]:
df2 = df1.withColumn(
    "sales_was_invalid",
    col("sales_num").isNull()
)


In [0]:
from pyspark.sql.functions import col, expr, coalesce, lit

df_final = df2.withColumn(
    "sales",
    coalesce(col("sales_num"), lit(0))
).drop("sales_num")


In [0]:

df_final.filter(col("sales_was_invalid")).count() 



In [0]:
df.count()

# Prectise

In [0]:
df=df_final

In [0]:
from pyspark.sql.functions import sum
df.select(sum("sales").alias("Total_sales"),
          sum("profit").alias("Total_profit")).show()

In [0]:
df.printSchema()

In [0]:
df.select("Customer ID").distinct().count()

In [0]:
df.filter(df.sales >500).show()

In [0]:
df.filter(df.Country=="India").show()

In [0]:
df.orderBy(df.sales.desc()).show()

In [0]:
from pyspark.sql.functions import col
df.filter(col("Ship Date") > col("Order Date")).show()


In [0]:
from pyspark.sql.functions import sum
df.groupBy("Region") \
      .agg(sum("sales").alias("total_Sales")) \
          .show()

In [0]:
df.groupby("category") \
    .agg(sum("profit").alias("total_profit")) \
    .show()

In [0]:
# from pyspark.sql.functions import (
#     col, expr, coalesce, lit,
#     avg, sum
# )

# # =========================
# # STEP 0: RAW DATA
# # =========================
# # maan lo ye tumhara raw dataframe hai
# df = spark.table("train_row")   # ya jo bhi tumhara table name ho


# # =========================
# # STEP 1: TEXT INVALID ROWS FLAG (OPTIONAL BUT GOOD)
# # =========================
# df1 = df.withColumn(
#     "sales_invalid_text",
#     ~col("sales").rlike("^[0-9.]+$")
# ).withColumn(
#     "discount_invalid_text",
#     ~col("discount").rlike("^[0-9.]+$")
# ).withColumn(
#     "profit_invalid_text",
#     ~col("profit").rlike("^[0-9.-]+$")
# )


# # =========================
# # STEP 2: SAFE CAST USING try_cast
# # =========================
# df2 = df1.withColumn(
#     "sales_num",
#     expr("try_cast(sales as double)")
# ).withColumn(
#     "discount_num",
#     expr("try_cast(discount as double)")
# ).withColumn(
#     "profit_num",
#     expr("try_cast(profit as double)")
# )


# # =========================
# # STEP 3: NULL -> 0 REPLACEMENT (BUSINESS RULE)
# # =========================
# df_final = df2.withColumn(
#     "sales",
#     coalesce(col("sales_num"), lit(0))
# ).withColumn(
#     "discount",
#     coalesce(col("discount_num"), lit(0))
# ).withColumn(
#     "profit",
#     coalesce(col("profit_num"), lit(0))
# ).drop(
#     "sales_num", "discount_num", "profit_num"
# )


# # =========================
# # STEP 4: VERIFICATION
# # =========================
# print("Total rows:", df_final.count())

# print("Invalid SALES rows fixed:",
#       df_final.filter(col("sales_invalid_text")).count())

# print("Invalid DISCOUNT rows fixed:",
#       df_final.filter(col("discount_invalid_text")).count())

# print("Invalid PROFIT rows fixed:",
#       df_final.filter(col("profit_invalid_text")).count())


# # =========================
# # STEP 5: SAFE AGGREGATIONS (NO ERROR)
# # =========================

# # Avg discount per segment
# df_final.groupBy("segment") \
#     .agg(avg("discount").alias("avg_discount")) \
#     .show()

# # Total sales & profit
# df_final.select(
#     sum("sales").alias("total_sales"),
#     sum("profit").alias("total_profit")
# ).show()


In [0]:
# from pyspark.sql.functions import avg
# df.groupBy("Segment") \
# .agg(avg("discount").alias("avg_discount")) \
# .show()

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Spark-Practice-Session") \
    .getOrCreate()

In [0]:
spark.version

In [0]:
dbutils.fs.ls("/Volumes/train/prectice_data/prectice/")


In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/train/prectice_data/prectice/part_1/*.csv")

df.show(5)
df.printSchema()


In [0]:
df.printSchema()

In [0]:
df.count()

In [0]:
df.columns

In [0]:
df.select("city").show(10)

In [0]:
from pyspark.sql.functions import expr
df=df.withColumn(
    "Sales",
    expr("try_cast(Sales as double)")
)

In [0]:
#df.createOrReplaceTempView("")

In [0]:
# df.printSchema()

In [0]:
df=df.withColumn("price",expr("try_cast(price as double)"))

In [0]:
# df.printSchema()

In [0]:
df=df.withColumn("discount",expr("try_cast(discount as double)"))

In [0]:
# df.select("price","quantity","discount").show(10)

In [0]:
from pyspark.sql.functions import regexp_replace ,col
df=df.withColumn("price",regexp_replace(col("price"),"[^0-9.]","")
                 .cast("double"))

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("price_clean",
                   regexp_replace(col("price"), "[^0-9.]", "").cast("double"))


In [0]:
df.select("price", "price_clean").show(10, truncate=False)
df.printSchema()


In [0]:
df=df.withColumn("discount_clean",col("discount").cast("double"))

In [0]:
df.select("price_clean","discount","discount_clean","quantity").show()

In [0]:
df.printSchema()

In [0]:
dbutils.fs.ls("/Volumes/train/prectice_data/prectice/part_1/")


In [0]:
dbutils.fs.head(
  "/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_j.json",
  1000
)


In [0]:
#Json file read 
df_json = spark.read.json("/Volumes/train/prectice_data/prectice/part_1/*.json")   #becouse json multiline array m h to error aa rhi ise read multiline optin k sath krna hoga

In [0]:
df_json = spark.read \
    .option("multiline", "true") \
    .json("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_j.json")


In [0]:
df_json.printSchema()
display(df_json)

In [0]:
df_json.show()

In [0]:
df.count()

In [0]:
df_json.count()

In [0]:
len(df.columns)

In [0]:
len(df_json.columns)

In [0]:
print("CSV columns :", len(df.columns))
print("JSON columns:", len(df_json.columns))


In [0]:
df.select("order_id").printSchema()

In [0]:
df_json.select("order_id").printSchema()

In [0]:
df.write \
    .mode("overwrite") \
    .parquet("/Volumes/train/prectice_data/prectice/parquet_output")

In [0]:
df_parquet = spark.read.parquet("/Volumes/train/prectice_data/prectice/parquet_output")

In [0]:
df_parquet.printSchema()
df_parquet.show(5, truncate=False)


In [0]:
df.count()

In [0]:
df_parquet.count()

In [0]:
df_json.printSchema()

In [0]:
df_parquet.columns

In [0]:
df_parquet.write \
    .mode("overwrite") \
        .option("header","true") \
        .csv("/Volumes/train/prectice_data/prectice/csv_from_parquet")

In [0]:
df_final = spark.read.parquet(
    "/Volumes/train/prectice_data/prectice/parquet_output"
)


In [0]:
df_final.groupBy("city").count().show()

In [0]:
df.show()

In [0]:
df.printSchema()

In [0]:
df.select("quantity","price_clean").printSchema()

In [0]:
for field in df_parquet.schema.fields:
    print(field.name,field.nullable)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
display(
    df_parquet.select([
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in df_parquet.columns
    ])
)

In [0]:
df_no_schema = spark.read \
    .option("header","true") \
    .option("inferSchema","false") \
    .csv("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_c.csv")
df_no_schema.printSchema()

In [0]:
df_infer = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_c.csv")
df_infer.printSchema()

In [0]:
dbutils.fs.ls("/Volumes/train/prectice_data/prectice/part_1/")

In [0]:
df_no_schema.printSchema()
df_infer.printSchema()

In [0]:
df_no_schema.withColumn("sales",col("quantity")*col("price")).show()

In [0]:
df_infer.withColumn("sales", col("quantity") * col("discount")).show()

In [0]:
from pyspark.sql.types import *
schema = StructType([
    StructField("order_id",IntegerType(),True),
    StructField("order_date",StringType(),True),
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("city", StringType(), True)
])

df_manual = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_c.csv")

df_manual.printSchema()


In [0]:
df_infer.printSchema()
df_manual.printSchema()

In [0]:
wrong_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("quantity", DoubleType(), True),   # galat
    StructField("price", IntegerType(), True)      # galat
])

df_wrong = spark.read \
    .option("header", "true") \
    .schema(wrong_schema) \
    .csv("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_c.csv")


In [0]:
df_json = spark.read.json("/Volumes/train/prectice_data/prectice/part_1/sales_transactions_sample_j.json")
df_json.printSchema()


In [0]:
dbutils.fs.ls("/Volumes/train/prectice_data/prectice/part_1/")


In [0]:
df_parquet = spark.read.parquet("/Volumes/train/prectice_data/prectice/part_1/transactions_sample.parquet")
df_parquet.printSchema()


In [0]:
df.show(5)

In [0]:
from pyspark.sql.functions import col
df.select("quantity").show()
df.select(col("quantity")).show()
df.select(df.quantity).show()

In [0]:
df_qty = df.select("quantity")
df_qty.show()

In [0]:
df.select("order_id","quantity","price").show()

In [0]:
df.select(col("quantity")+10).show()

In [0]:
df.select(col("quantity") * col("price_clean")).show()

In [0]:
sales_expr = col("quantity").cast("int") * col("price_clean").cast("double")
df.select(sales_expr.alias("sales")).show()

In [0]:
df.select(col("quantitty")).show()

In [0]:
df.select("Quantity").show()

In [0]:
df.select(col("quantity").alias("qty")).show()

In [0]:
df.select(
    ((col("quantity").cast("int") * col("price_clean").cast("double") +100) .alias("final_amount"))
).show()

In [0]:
df.drop(col("discount")).show()

In [0]:
df.drop("discount").show()

In [0]:
sales_expr=col("Quantity").cast("int") * col("price_clean").cast("double")
df.select("order_id",sales_expr.alias("salesss")).show()
df.withColumn("sales",sales_expr).show()

In [0]:
df.head()

In [0]:
df.take(5)

In [0]:
print(type(df.head()))
print(type(df.take(5)))

In [0]:
rows=df.take(5)
for r in rows:
    print(r.order_id,r.quantity)

In [0]:
df.show(5)
df.show(5)
df.show(5)

In [0]:
df.head(1)


In [0]:
df.head()

In [0]:
df.explain()

In [0]:
df.collect()

In [0]:
df.cache()
df.count()


In [0]:
df.columns

In [0]:
len(df.columns)

In [0]:
for i,c in enumerate(df.columns):
    print(i,c)

In [0]:
df_renamed = df.withColumnRenamed("price", "unit_price")

df_renamed.columns


In [0]:
df_dropped =df.drop("discount")
len(df_dropped.columns)

In [0]:
df.columns


In [0]:
df_json.columns

In [0]:
df_parq.columns

In [0]:
if "ordery_date" in df.columns:
    print("Column exists")
else:
    print("Column missing")

In [0]:
from pyspark.sql.functions import lit
df_new = df.withColumn("data_Source",lit("csv"))
df_new.columns

In [0]:
r_col = ["order_id","customer_id","price"]
e_col=[c for c in r_col if c in df.columns]
df.select(e_col).collect()

In [0]:
df.show(5)

In [0]:
from pyspark.sql.functions import lit
df1 = df.withColumn("source", lit("csv"))


In [0]:
df2 = df1.withColumn("country", lit("India"))

In [0]:
df2.show()

In [0]:
from pyspark.sql.functions import col
qty_expr = col("quantity").cast("int")

In [0]:
from pyspark.sql.functions import regexp_replace
price_clean_expr = regexp_replace(col("price"), "₹", "")


In [0]:
df.select(col("order_id"))

In [0]:
df3 = df2.withColumn(
    "row_amount",
    col("price")* col("quantity")
)

In [0]:
df4=df3.withColumn("remarks",lit(None))

In [0]:
from pyspark.sql.functions import upper
df5 = df4.withColumn("city_upper",upper(col("city")))

In [0]:
df5.show()

In [0]:
df6 = (
    df
    .withColumn("source", lit("csv"))
    .withColumn("country", lit("India"))
    .withColumn("city_upper", upper(col("city")))
)


In [0]:
df.withColumn("test",lit("csv"))

In [0]:
amount_expr = (
    regexp_replace(col("price"), "₹", "").cast("double")
    * col("quantity").cast("int"))
df7=df.withColumn("amount",amount_expr) \
        .withColumn("amount_with_tax",amount_expr +100)


In [0]:
df7.printSchema()

In [0]:
df7.limit(5).show()